In [1]:
from DDCASPT2 import DDCASPT2
import pickle, os, shutil
from glob import glob
import numpy as np
from joblib import Parallel, delayed
import pandas as pd

In [2]:
radius_range=np.linspace(0.6,3,100)
# radius_range=[0.94]
chains=np.arange(2,14,2)
# chains=np.arange(14,22,2)
print(chains)
# train_ind,test_ind=train_test_split(radius_range, test_size=0.3, random_state=0)
# print(len(train_ind),len(test_ind))
# with open('train_ind.pickle', 'wb') as handle:
#     pickle.dump(train_ind, handle, protocol=pickle.HIGHEST_PROTOCOL)

# with open('test_ind.pickle', 'wb') as handle:
#     pickle.dump(test_ind, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('test_ind.pickle', 'rb') as handle:
    test_ind = pickle.load(handle)

with open('train_ind.pickle', 'rb') as handle:
    train_ind = pickle.load(handle)
    
print(len(train_ind),len(test_ind))    


[ 2  4  6  8 10 12]
70 30


/tmp/ipykernel_307783/2467865580.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  test_ind = pickle.load(handle)
/tmp/ipykernel_307783/2467865580.py:18: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is 

In [3]:
# for i in chains:
topdir = os.getcwd()
basis_set='ANO-RCC-VDZP'


In [4]:
# def run(i):
#     dirname=f'H{i}_chain'
#     print(dirname)
#     if os.path.exists(dirname)==False:
#         os.mkdir(dirname)
        
#     # os.chdir(os.path.join(topdir,dirname))    
#     for idxr, r in enumerate(radius_range):
        
#         # Loop radius
#         name=f"H{i}_{r:.2f}"
        
#         # Create files
#         subdirpath = os.path.join(topdir,dirname,f'{name}')
#         if os.path.exists(subdirpath)==False:
#             os.mkdir(subdirpath)
#         if os.path.exists(os.path.join(subdirpath,f'{name}.csv'))==False:
#             shutil.rmtree(os.path.join('tmp'),name)
            
#             # Write xyz
#             with open(os.path.join(subdirpath,f'{name}.xyz'),'w') as f:
#                 f.write(f'{i}\n\n')
#                 for j in range(i):
#                     f.write(f'H {0:>8f} {0:>8f} {j*r:>8f}\n')
#             print(subdirpath)
#             if idxr==0:
#                 d = DDCASPT2(subdirpath,basis_set,name,i,i,0,previous=None)()
#             else:            
#                 previous=os.path.join(topdir,dirname,f'H{i}_{radius_range[idxr-1]:.2f}',f"H{i}_{radius_range[idxr-1]:.2f}.RasOrb")
#                 print(previous)
#                 d = DDCASPT2(subdirpath,basis_set,name,i,i,0,previous=previous)()
    
#     for idxr, r in enumerate(radius_range):
#         # Loop radius
#         name=f"H{i}_{r:.2f}" 
        
#         subdirpath = os.path.join(topdir,dirname,f'{name}')
        
        
#         for j in glob(os.path.join(subdirpath,"*GMJ*.csv"))+glob(os.path.join(subdirpath,"*Orb*"))+glob(os.path.join(subdirpath,"*h5"))+glob(os.path.join(subdirpath,"xmldump")):
#             os.remove(j)        




def run(i):
    dirname = f'H{i}_chain'
    print(dirname)
    
    if not os.path.exists(dirname):
        os.mkdir(dirname)
        
    first_valid_idx = None  # Track the first valid index
    
    for idxr, r in enumerate(radius_range):
        # Loop radius
        name = f"H{i}_{r:.2f}"
        
        # Create files
        subdirpath = os.path.join(topdir, dirname, f'{name}')
        if not os.path.exists(subdirpath):
            os.mkdir(subdirpath)
        
        if not os.path.exists(os.path.join(subdirpath, f'{name}_1.csv')):
            shutil.rmtree(os.path.join('tmp'), ignore_errors=True)
            
            # Write xyz
            with open(os.path.join(subdirpath, f'{name}.xyz'), 'w') as f:
                f.write(f'{i}\n\n')
                for j in range(i):
                    f.write(f'H {0:>8f} {0:>8f} {j*r:>8f}\n')
            
            print(subdirpath)

            try:
                if first_valid_idx is None:  # First attempt
                    d = DDCASPT2(subdirpath, basis_set, name, i, i, 0, casscf_previous=None)()
                    first_valid_idx = idxr  # Mark this as the first valid calculation
                else:
                    previous = os.path.join(
                        topdir, dirname, f'H{i}_{radius_range[first_valid_idx]:.2f}', 
                        f"H{i}_{radius_range[first_valid_idx]:.2f}.RasOrb"
                    )
                    print(previous)
                    d = DDCASPT2(subdirpath, basis_set, name, i, i, 0, casscf_previous=previous)()
            except Exception as e:
                print(f"Error at index {idxr} with r={r}: {e}")
                continue  # Skip to the next iteration if it fails
    
    # Cleanup section
    for idxr, r in enumerate(radius_range):
        name = f"H{i}_{r:.2f}" 
        subdirpath = os.path.join(topdir, dirname, f'{name}')
        
        for j in glob(os.path.join(subdirpath, "*GMJ*.csv")) + \
                 glob(os.path.join(subdirpath, "*Orb*")) + \
                 glob(os.path.join(subdirpath, "*h5")) + \
                 glob(os.path.join(subdirpath, "xmldump")):
            os.remove(j)


In [ ]:
Parallel(n_jobs=-1)(delayed(run)(i) for i in chains)

H6_chain
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H6_chain/H6_2.81
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'S1': 6, 'S2': 7, 'S3': 8, 'S4': 9, 'S5': 10, 'S6': 11, 'S7': 12, 'S8': 13, 'S9': 14, 'S10': 15, 'S11': 16, 'S12': 17, 'S13': 18, 'S14': 19, 'S15': 20, 'S16': 21, 'S17': 22, 'S18': 23, 'S19': 24, 'S20': 25, 'S21': 26, 'S22': 27, 'S23': 28, 'S24': 29}


Pairs: 100%|██████████| 3/3 [00:00<00:00, 20.61it/s]

Features:  56%|█████▌    | 20/36 [00:01<00:01, 12.80it/s]

H8_chain
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H8_chain/H8_2.81
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'A7': 6, 'A8': 7, 'S1': 8, 'S2': 9, 'S3': 10, 'S4': 11, 'S5': 12, 'S6': 13, 'S7': 14, 'S8': 15, 'S9': 16, 'S10': 17, 'S11': 18, 'S12': 19, 'S13': 20, 'S14': 21, 'S15': 22, 'S16': 23, 'S17': 24, 'S18': 25, 'S19': 26, 'S20': 27, 'S21': 28, 'S22': 29, 'S23': 30, 'S24': 31, 'S25': 32, 'S26': 33, 'S27': 34, 'S28': 35, 'S29': 36, 'S30': 37, 'S31': 38, 'S32': 39}


Pairs: 100%|██████████| 3/3 [00:00<00:00,  6.92it/s]

Root: 100%|██████████| 1/1 [00:03<00:00,  3.07s/it]

Pairs: 100%|██████████| 3/3 [00:00<00:00, 19.25it/s]

Features:   0%|          | 0/36 [00:00<?, ?it/s]

/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H6_chain/H6_2.83
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H6_chain/H6_2.81/H6_2.81.RasOrb
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'S1': 6, 'S2': 7, 'S3': 8, 'S4': 9, 'S5': 10, 'S6': 11, 'S7': 12, 'S8': 13, 'S9': 14, 'S10': 15, 'S11': 16, 'S12': 17, 'S13': 18, 'S14': 19, 'S15': 20, 'S16': 21, 'S17': 22, 'S18': 23, 'S19': 24, 'S20': 25, 'S21': 26, 'S22': 27, 'S23': 28, 'S24': 29}


...


Features:  51%|█████     | 51/100 [00:50<00:44,  1.10it/s]

H2_chain



Root: 100%|██████████| 1/1 [01:39<00:00, 99.23s/it]


/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H10_chain/H10_2.85
/home/grierjones/DDCASPT2/hydrogen_comps/rerun/H10_chain/H10_2.81/H10_2.81.RasOrb
Running on None cores
Found a valid MOLCAS installation at /home/grierjones/Test/build
MOLCAS_WORKDIR is set to /tmp/
{'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'A7': 6, 'A8': 7, 'A9': 8, 'A10': 9, 'S1': 10, 'S2': 11, 'S3': 12, 'S4': 13, 'S5': 14, 'S6': 15, 'S7': 16, 'S8': 17, 'S9': 18, 'S10': 19, 'S11': 20, 'S12': 21, 'S13': 22, 'S14': 23, 'S15': 24, 'S16': 25, 'S17': 26, 'S18': 27, 'S19': 28, 'S20': 29, 'S21': 30, 'S22': 31, 'S23': 32, 'S24': 33, 'S25': 34, 'S26': 35, 'S27': 36, 'S28': 37, 'S29': 38, 'S30': 39, 'S31': 40, 'S32': 41, 'S33': 42, 'S34': 43, 'S35': 44, 'S36': 45, 'S37': 46, 'S38': 47, 'S39': 48, 'S40': 49}


Pairs: 100%|██████████| 3/3 [00:01<00:00,  2.94it/s]

Root: 100%|██████████| 1/1 [01:27<00:00, 87.34s/it]
